In [ ]:
# -- Cell 1 -- rclone + Drive. Same pattern as the previous two notebooks.# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, and the# RCLONE_DRIVE_TOKEN secret attached to THIS notebook.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE

In [ ]:
# -- Cell 2 -- deps + GPU.subprocess.run('pip install -q -U "transformers>=5.0" pyarrow', shell=True, check=True)subprocess.run("pip uninstall -y -q torchao", shell=True)   # same peft/torchao clash guardimport torch, numpy as np, pandas as pd, glob, json, timeprint("torch", torch.__version__, "| GPUs", torch.cuda.device_count())for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))assert torch.cuda.is_available(), "this notebook needs a GPU"# fp16 was verified safe for this model on padded batches: their attention builds# (1 - mask) * -1e9, which overflows fp16 (max 65504), BUT the mask is cast to# float32 before scaled_dot_product_attention so torch promotes correctly.# Measured: 0 NaN / 0 inf, and mean_pool differs from fp32 by 3.3e-3.

In [ ]:
# -- Cell 3 -- inputs: the 2M subset and the teacher.# Subset comes from the Kaggle Dataset if it is attached (instant), otherwise off# Drive. Teacher always off Drive.SUB = Nonefor cand in glob.glob("/kaggle/input/*/**/pretrain_subset_2M.parquet", recursive=True):    SUB = cand; breakif SUB is None:    os.makedirs("/kaggle/working/subset", exist_ok=True)    subprocess.run("rclone copy %s/data/pretrain_subset_2M /kaggle/working/subset -P" % REMOTE,                   shell=True, check=True)    SUB = "/kaggle/working/subset/pretrain_subset_2M.parquet"print("subset:", SUB)TEACH = "/kaggle/working/models/peptideclm-2-mlm-large"if not os.path.exists(TEACH + "/model.safetensors"):    snaps = "%s/models/models--aaronfeller--peptideclm-2-mlm-large/snapshots" % REMOTE    sha = subprocess.run("rclone lsf " + snaps, shell=True, capture_output=True,                         text=True).stdout.split()[0].rstrip("/")    os.makedirs(TEACH, exist_ok=True)    subprocess.run("rclone copy %s/%s %s -P" % (snaps, sha, TEACH), shell=True, check=True)meta = pd.read_parquet(SUB, columns=["source", "smiles", "n_tokens"])print("molecules {:,} | tokens {:,}".format(len(meta), int(meta.n_tokens.sum())))print(meta.source.value_counts().to_string())print("teacher:", TEACH, "%.2f GB" % (os.path.getsize(TEACH + "/model.safetensors") / 1e9))

In [ ]:
# -- Cell 4 -- unit-test the mask generator before writing the worker.## VERBATIM port of their pretraining.py:202-233 -- per-sequence 25% budget, spans# ~ N(3.5, 1.0) floored at 1, overshooting spans truncated so the budget lands# exactly, and an overlap check that rejects a span by setting end_pos = start_pos.# Masking happens BEFORE [CLS]/[SEP] are added (they tokenize with# add_special_tokens=False), so those two are never masked.## ONE DELIBERATE DEVIATION: their while-loop has no escape. If no non-overlapping# span can be placed it spins forever. We add a guard. Replicating that faithfully# would hang the run.## TWO MASKS: A, then B drawn from the positions A did not take. Together they# supervise 50% of every molecule. Verified achievable -- B hits the full 25% at# every length from 10 to 512 tokens with zero overlap.MASK_PCT, MU, SD = 0.25, 3.5, 1.0def span_mask_positions(seq_len, rng, forbidden=None, pct=MASK_PCT, mu=MU, sd=SD,                        guard_mult=50):    n_target = int(seq_len * pct)    forb = forbidden or set()    masked, count, guard = set(), 0, 0    while count < n_target and guard < guard_mult * max(1, n_target):        guard += 1        span = max(1, int(rng.normal(mu, sd)))        if count + span > n_target:            span = n_target - count        start = int(rng.integers(0, seq_len))        end = min(seq_len, start + span)        for pos in range(start, end):            if pos in masked or pos in forb:                end = start                break        masked.update(range(start, end))        count += (end - start)    return sorted(masked)rng = np.random.default_rng(0)print("%6s %8s %16s %16s %9s" % ("len", "target", "A achieved", "B achieved", "overlap"))for L in [10, 50, 158, 340, 512]:    a, b, ov = [], [], 0    for _ in range(200):        A = span_mask_positions(L, rng)        B = span_mask_positions(L, rng, forbidden=set(A))        a.append(len(A)); b.append(len(B)); ov += len(set(A) & set(B))    print("%6d %8d %8.1f (%4.1f%%) %8.1f (%4.1f%%) %9d"          % (L, int(L * MASK_PCT), np.mean(a), 100 * np.mean(a) / L,             np.mean(b), 100 * np.mean(b) / L, ov))    assert ov == 0, "masks overlap"print("\nmask generator OK -- A and B disjoint")

In [ ]:
# -- Cell 5 -- write the worker. One process per GPU; cell 7 launches both.## WHAT IT CACHES, and why:#   top-16 LOG-PROBS + idx  -> soft MLM KD. topv holds log p = logit - lse, so#                              p = exp(topv). Storing RAW logits in fp16 was tried#                              and rejected: at magnitude ~20 fp16 resolution is#                              ~0.016 and exp() turns that into sum(p) = 1.0075,#                              i.e. a NEGATIVE residual. Log-probs sit near 0 where#                              fp16 is fine: max sum 1.0003, max error 1.5e-4.#                              Measured on the teacher: top-16 holds 96.2% of the#                              mass on average, 80% at p05; top-8 falls to 63% at#                              p05, exactly where soft targets carry information.#   logsumexp (full 405)    -> raw logits stay recoverable (logit = topv + lse) and#                              the tail mass is known, so the KD loss can normalise#                              honestly instead of pretending the tail is zero.#   mean_pool (unmasked)    -> SPKD. Computed on the CLEAN sequence: the geometry we#                              want to transfer is the teacher's view of the#                              molecule, not of a corrupted copy.#   masked positions        -> stored explicitly, not as an RNG seed, so targets can#                              never drift out of alignment if numpy's generator#                              changes.## THREE forward passes per batch, not four: clean (mean_pool) + mask A + mask B.# mean_pool does not depend on the mask, so the second mask costs +50%, not +100%.## Descriptors are NOT cached -- already in the parquet, no teacher needed.WORKER = "/kaggle/working/cache_worker.py"open(WORKER, "w").write(r"""import argparse, os, numpy as np, pandas as pd, torchimport torch.nn.functional as Ffrom transformers import AutoTokenizer, AutoModelMASK_PCT, MU, SD, TOPK = 0.25, 3.5, 1.0, 16# ---------------------------------------------------------------------------# Their MultiHeadAttention hardcodes the padding mask to float32:#     mask = mask.to(dtype=torch.float32); mask = (1.0 - mask) * -1e9# and hands it to scaled_dot_product_attention. With the model in fp16 the# queries are fp16 and CUDA refuses:#     RuntimeError: invalid dtype for bias - should match query's dtype# (CPU silently promotes, which is why this only shows up on GPU.)## Wrapping SDPA is the least invasive fix: their model file is untouched, and we# keep fp16 -- running fp32 on a T4 would be 3-4x slower and turn a 5 h job into# ~20 h. The clamp matters too: -1e9 cast to fp16 becomes -inf, so we pin it to# the largest finite negative instead and avoid inf arithmetic entirely._orig_sdpa = F.scaled_dot_product_attentiondef _sdpa_dtype_safe(query, key, value, attn_mask=None, **kw):    if attn_mask is not None and attn_mask.dtype not in (torch.bool, query.dtype):        attn_mask = attn_mask.to(query.dtype).clamp_(min=torch.finfo(query.dtype).min)    return _orig_sdpa(query, key, value, attn_mask=attn_mask, **kw)F.scaled_dot_product_attention = _sdpa_dtype_safe# ---------------------------------------------------------------------------def span_mask_positions(seq_len, rng, forbidden=None, guard_mult=50):    n_target = int(seq_len * MASK_PCT)    forb = forbidden or set()    masked, count, guard = set(), 0, 0    while count < n_target and guard < guard_mult * max(1, n_target):        guard += 1        span = max(1, int(rng.normal(MU, SD)))        if count + span > n_target:            span = n_target - count        start = int(rng.integers(0, seq_len))        end = min(seq_len, start + span)        for pos in range(start, end):            if pos in masked or pos in forb:                end = start                break        masked.update(range(start, end))        count += (end - start)    return sorted(masked)def main():    ap = argparse.ArgumentParser()    ap.add_argument('--subset', required=True); ap.add_argument('--teacher', required=True)    ap.add_argument('--out', required=True)    ap.add_argument('--rank', type=int, default=0); ap.add_argument('--world', type=int, default=1)    ap.add_argument('--shard-size', type=int, default=50000)    ap.add_argument('--max-tokens', type=int, default=16384)    ap.add_argument('--limit', type=int, default=0)    a = ap.parse_args()    os.makedirs(a.out, exist_ok=True)    tok = AutoTokenizer.from_pretrained(a.teacher, trust_remote_code=True)    CLS, SEP, PAD, MSK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id    model = AutoModel.from_pretrained(a.teacher, trust_remote_code=True,                                      use_safetensors=True).eval().half().cuda()    DIM = model.config.embed_dim    # Prove the SDPA patch works on a padded fp16 batch before doing real work.    with torch.no_grad():        _ids = torch.tensor([[CLS, 10, 11, SEP, PAD], [CLS, 12, SEP, PAD, PAD]]).cuda()        _att = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 1, 0, 0]]).cuda()        _o = model(input_ids=_ids, attention_mask=_att)        assert torch.isfinite(_o.logits).all() and torch.isfinite(_o.mean_pool).all(), \            'fp16 forward produced non-finite values'    print('[r%d] fp16 padded-batch check OK' % a.rank, flush=True)    smiles = pd.read_parquet(a.subset, columns=['smiles']).smiles.values    n = len(smiles) if a.limit == 0 else min(a.limit, len(smiles))    starts = list(range(0, n, a.shard_size))    for si, s0 in enumerate(starts):        if si % a.world != a.rank:            continue        path = '%s/shard_%05d.npz' % (a.out, si)        if os.path.exists(path):            print('[r%d] shard %d exists, skip' % (a.rank, si), flush=True); continue        s1 = min(s0 + a.shard_size, n)        enc = tok(list(smiles[s0:s1]), add_special_tokens=False)['input_ids']        # Masks in pre-special coords, shifted +1 for the [CLS] we prepend.        rng = np.random.default_rng(1234 + si)        seqs, mp_a, mp_b = [], [], []        for ids in enc:            A = span_mask_positions(len(ids), rng)            B = span_mask_positions(len(ids), rng, forbidden=set(A))            seqs.append(ids)            mp_a.append([p + 1 for p in A]); mp_b.append([p + 1 for p in B])        # Length-bucketed batching. The corpus is bimodal (PubChem ~23 tokens,        # peptides ~340); naive batching pads short molecules ~20x.        order = np.argsort([len(s) for s in seqs])        batches, cur, curmax = [], [], 0        for i in order:            L = len(seqs[i]) + 2            if cur and (max(curmax, L) * (len(cur) + 1) > a.max_tokens):                batches.append(cur); cur, curmax = [i], L            else:                cur.append(i); curmax = max(curmax, L)        if cur:            batches.append(cur)        MP = np.zeros((len(seqs), DIM), dtype=np.float16)        # Slots are indexed BY MOLECULE, not appended in batch order. Batches are        # length-sorted, so appending would concatenate the data in a different        # order than the ptr cumsum below assumes -- every molecule's targets would        # silently point at another molecule's positions.        acc = {k: {'v': [None] * len(seqs), 'i': [None] * len(seqs),                   'l': [None] * len(seqs), 'p': [None] * len(seqs),                   'c': np.zeros(len(seqs), dtype=np.int64)}               for k in ('a', 'b')}        with torch.no_grad():            for bi, idxs in enumerate(batches):                T = max(len(seqs[i]) for i in idxs) + 2                ids = np.full((len(idxs), T), PAD, dtype=np.int64)                att = np.zeros((len(idxs), T), dtype=np.int64)                for r, i in enumerate(idxs):                    sq = seqs[i]                    ids[r, 0] = CLS; ids[r, 1:1 + len(sq)] = sq; ids[r, 1 + len(sq)] = SEP                    att[r, :len(sq) + 2] = 1                ids_t = torch.from_numpy(ids).cuda(); att_t = torch.from_numpy(att).cuda()                # pass 1: clean -> mean_pool (mask-independent, computed once)                MP[idxs] = model(input_ids=ids_t, attention_mask=att_t                                 ).mean_pool.float().cpu().numpy().astype(np.float16)                # passes 2 and 3: mask A, then mask B                for key, mpos in (('a', mp_a), ('b', mp_b)):                    rows, cols = [], []                    for r, i in enumerate(idxs):                        for p in mpos[i]:                            rows.append(r); cols.append(p)                    if not rows:                        continue                    rt = torch.tensor(rows).cuda(); ct = torch.tensor(cols).cuda()                    mids = ids_t.clone(); mids[rt, ct] = MSK                    lg = model(input_ids=mids, attention_mask=att_t).logits.float()                    sel = lg[rt, ct]                    lse = torch.logsumexp(sel, dim=-1)                    v, ix = sel.topk(TOPK, dim=-1)                    v = v - lse[:, None]              # store LOG-PROBS, see cell comment                    v = v.cpu().numpy(); ix = ix.cpu().numpy(); lse = lse.cpu().numpy()                    rr = np.array(rows); off = 0                    for r, i in enumerate(idxs):                        k = int((rr == r).sum())                        if k == 0:                            continue                        acc[key]['v'][i] = v[off:off + k]                        acc[key]['i'][i] = ix[off:off + k]                        acc[key]['l'][i] = lse[off:off + k]                        acc[key]['p'][i] = np.array(mpos[i], dtype=np.int16)                        acc[key]['c'][i] = k; off += k                if bi % 50 == 0:                    print('[r%d] shard %d batch %d/%d' % (a.rank, si, bi, len(batches)), flush=True)        out = {'mol_idx': np.arange(s0, s1, dtype=np.int64), 'mean_pool': MP}        for key in ('a', 'b'):            d = acc[key]            # Concatenate in MOLECULE order so the cumsum offsets are meaningful.            keepi = [i for i in range(len(seqs)) if d['c'][i] > 0]            def cat(field, dtype, width=None):                if not keepi:                    return np.zeros((0,) if width is None else (0, width), dtype=dtype)                return np.concatenate([d[field][i] for i in keepi]).astype(dtype)            out['ptr_' + key] = np.concatenate([[0], np.cumsum(d['c'])]).astype(np.int64)            out['pos_' + key] = cat('p', np.int16)            out['topv_' + key] = cat('v', np.float16, TOPK)            out['topi_' + key] = cat('i', np.int16, TOPK)            out['lse_' + key] = cat('l', np.float32)            assert out['ptr_' + key][-1] == len(out['pos_' + key])        tmp = path + '.tmp.npz'        np.savez(tmp, **out)        os.replace(tmp, path)          # atomic: a killed run leaves no half shard        print('[r%d] shard %d done: %d mols, A %d / B %d positions, %.1f MB'              % (a.rank, si, s1 - s0, int(acc['a']['c'].sum()), int(acc['b']['c'].sum()),                 os.path.getsize(path) / 1e6), flush=True)if __name__ == '__main__':    main()""")print("wrote", WORKER)

In [ ]:
# -- Cell 6 -- smoke test on 2,000 molecules before committing hours.CACHE = "/tmp/cache"          # NOT /kaggle/working: 15 GB of shards would crowd the                              # 20 GB quota alongside the model and subset. /tmp sits                              # on the larger disk. Cell 8 ships it to Drive.SMOKE = "/tmp/cache_smoke"r = subprocess.run(["python", "-u", WORKER, "--subset", SUB, "--teacher", TEACH,                    "--out", SMOKE, "--shard-size", "1000", "--limit", "2000"],                   env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))assert r.returncode == 0, "smoke failed"z = np.load(sorted(glob.glob(SMOKE + "/*.npz"))[0])print("\narrays:", {k: (z[k].shape, str(z[k].dtype)) for k in z.files})real = meta.n_tokens[:1000].sub(2).sum()for key in ("a", "b"):    p = np.exp(z["topv_" + key].astype(np.float32))    tot = p.sum(1)    print("mask %s: %d positions (%.4f of tokens) | top-16 mass mean %.5f p05 %.5f max %.6f"          % (key.upper(), z["ptr_" + key][-1], z["ptr_" + key][-1] / real,             tot.mean(), np.percentile(tot, 5), tot.max()))    assert tot.max() <= 1.001, "top-k mass out of tolerance -- storage is wrong"    assert (z["pos_" + key] >= 1).all(), "[CLS] was masked"# Per molecule: A and B must be disjoint, and every position must fall inside# that molecule's own length. The length check is what catches a CSR ordering bug# -- misaligned slices show up as positions past the end of a short molecule.lens = meta.n_tokens.values[:len(z["mol_idx"])]bad_ov = bad_range = 0for i in range(len(z["mol_idx"])):    A = set(z["pos_a"][z["ptr_a"][i]:z["ptr_a"][i + 1]].tolist())    B = set(z["pos_b"][z["ptr_b"][i]:z["ptr_b"][i + 1]].tolist())    bad_ov += len(A & B)    bad_range += sum(1 for p in (A | B) if p > lens[i] - 2)assert bad_ov == 0, "masks A and B overlap"assert bad_range == 0, "positions outside the molecule -- ptr/data ordering is wrong"print("\nA and B disjoint across all molecules | mean_pool finite:",      bool(np.isfinite(z["mean_pool"]).all()))import shutil; shutil.rmtree(SMOKE, ignore_errors=True)print("smoke OK")

In [ ]:
# -- Cell 7 -- the real run: one worker per GPU, resumable across sessions.## Shards already on Drive are pulled back first, so a session that dies mid-run# resumes instead of restarting. Three teacher passes over 2M molecules is roughly# 5-7 h on T4 x2 -- close enough to the session limit that this matters. Run it as# a batch commit (Save Version -> Save & Run All).import timeos.makedirs(CACHE, exist_ok=True)DEST = REMOTE + "/data/teacher_cache_2M"print("pulling any existing shards back from Drive to resume...")subprocess.run("rclone copy %s %s --include '*.npz' --transfers 8 -P" % (DEST, CACHE),               shell=True, check=False)print("resuming with %d shards already done\n" % len(glob.glob(CACHE + "/*.npz")))NGPU = torch.cuda.device_count()SHARD = 50000TOTAL = int(np.ceil(len(meta) / SHARD))procs = []for rank in range(NGPU):    log = open("/tmp/cache_r%d.log" % rank, "w")    p = subprocess.Popen(["python", "-u", WORKER, "--subset", SUB, "--teacher", TEACH,                          "--out", CACHE, "--rank", str(rank), "--world", str(NGPU),                          "--shard-size", str(SHARD)],                         stdout=log, stderr=subprocess.STDOUT,                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(rank)))    procs.append((rank, p)); print("launched rank %d on GPU %d" % (rank, rank))# Mirror to Drive every 10 min so a killed session keeps finished shards.SYNC = subprocess.Popen("while true; do rclone copy %s %s --include '*.npz' "                        "--drive-chunk-size 64M >> /tmp/rclone_sync.log 2>&1; sleep 600; done"                        % (CACHE, DEST), shell=True)print("background sync -> Drive every 10 min\n")t0 = time.time()while any(p.poll() is None for _, p in procs):    time.sleep(120)    done = len(glob.glob(CACHE + "/*.npz"))    gb = sum(os.path.getsize(f) for f in glob.glob(CACHE + "/*.npz")) / 1e9    el = (time.time() - t0) / 60    eta = (el / max(done, 1) * (TOTAL - done))    print("[%6.1f min] shards %d/%d  %.2f GB  ETA %.0f min" % (el, done, TOTAL, gb, eta))# Stop the mirror before Cell 8's final copy, or two rclone processes write the# same Drive directory at once.SYNC.terminate()subprocess.run("pkill -f 'rclone copy %s' || true" % CACHE, shell=True)for rank, p in procs:    print("rank %d exit %d" % (rank, p.returncode))    if p.returncode != 0:        print(open("/tmp/cache_r%d.log" % rank).read()[-3000:])print("\nelapsed %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# -- Cell 8 -- verify the whole cache, write the manifest, ship it.files = sorted(glob.glob(CACHE + "/*.npz"))n_mol = 0; n_pos = {"a": 0, "b": 0}; masses = {"a": [], "b": []}gb = sum(os.path.getsize(f) for f in files) / 1e9for f in files:    z = np.load(f)    n_mol += len(z["mol_idx"])    for k in ("a", "b"):        n_pos[k] += int(z["ptr_" + k][-1])        if len(masses[k]) < 20:            masses[k].append(np.exp(z["topv_" + k][:5000].astype(np.float32)).sum(1))real_tokens = int(meta.n_tokens.sum() - 2 * len(meta))print("shards %d | molecules %s | %.2f GB" % (len(files), "{:,}".format(n_mol), gb))for k in ("a", "b"):    m = np.concatenate(masses[k])    print("mask %s: %s positions (%.4f of tokens) | top-16 mass mean %.5f p05 %.5f"          % (k.upper(), "{:,}".format(n_pos[k]), n_pos[k] / real_tokens,             m.mean(), np.percentile(m, 5)))assert n_mol == len(meta), "cache incomplete -- re-run Cell 7, it resumes"manifest = {    "teacher": "peptideclm-2-mlm-large",    "n_molecules": int(n_mol),    "n_masked_positions": {k: int(v) for k, v in n_pos.items()},    "mask": {"pct": MASK_PCT, "span_mu": MU, "span_sd": SD, "n_masks": 2,             "disjoint": True, "seed": "default_rng(1234 + shard_index)",             "note": "A and B are disjoint 25% masks covering 50% of each molecule. "                     "Alternate by epoch (epoch 0 -> A, epoch 1 -> B) rather than "                     "sampling randomly: with 2 epochs, random choice leaves half "                     "the molecules seeing one mask twice."},    "topk": 16,    "format": {"topv_{a,b}": "log-probabilities (logit - logsumexp), fp16 -- p = exp(topv)",               "topi_{a,b}": "vocab indices, int16",               "lse_{a,b}": "full-vocab logsumexp, fp32 -- raw logit = topv + lse",               "ptr_{a,b}": "CSR offsets: molecule i owns pos[ptr[i]:ptr[i+1]]",               "pos_{a,b}": "masked positions, final coords ([CLS] at 0, so pos >= 1)",               "mean_pool": "fp16, UNMASKED forward, shared by both masks",               "residual": "1 - sum(exp(topv)); CLAMP AT 0 -- fp16 can round it "                           "a hair negative on near-certain predictions"},    "not_cached": {"descriptors": "already in the subset parquet, pre-normalized",                   "hidden_states": "656 GB/layer -- FitNets needs a live teacher"},    "size_gb": round(gb, 2),}json.dump(manifest, open(CACHE + "/manifest.json", "w"), indent=2)print("\n" + json.dumps(manifest, indent=2))subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (CACHE, DEST),               shell=True, check=True)print("\nuploaded to " + DEST)